<a href="https://colab.research.google.com/github/Kusinga321/Nvidia-interview/blob/main/copy_transpose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile transpose.cu
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <cuda_runtime.h>

#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  printf("CUDA error %s (%s:%d)\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)

#define TILE 32
using KernelFn = void (*)(const float*, float*, int, int);

// in: macierz height x width (row-major), out: ten sam rozmiar
__global__ void copy_kernel(const float* in, float* out, int width, int height) {
    int thr1 = blockIdx.x * blockDim.x + threadIdx.x;
    int thr2 = blockIdx.y * blockDim.y + threadIdx.y;
    if (thr2 >= height || thr1 >= width)
        return;

    out[thr2 * width + thr1] = in[thr2 * width + thr1];
}

// in: height x width, out: width x height
__global__ void transpose_naive(const float* in, float* out, int width, int height) {
    int thr1 = blockIdx.x * blockDim.x + threadIdx.x;
    int thr2 = blockIdx.y * blockDim.y + threadIdx.y;
    if (thr2 >= height || thr1 >= width)
        return;

    out[thr1 * height + thr2] = in[thr2 * width + thr1];
}

// in: height x width, out: width x height
__global__ void transpose_tiled(const float* in, float* out, int width, int height) {
    __shared__ float tile[TILE][TILE+1];

    int x = blockIdx.x * TILE + threadIdx.x;        // kolumna w in
    int y = blockIdx.y * TILE + threadIdx.y;        // wiersz w in
    if (x < width && y < height)
        tile[threadIdx.y][threadIdx.x] = in[y * width + x];
    __syncthreads();

    int ox = blockIdx.y * TILE + threadIdx.x;       // kolumna w out
    int oy = blockIdx.x * TILE + threadIdx.y;       // wiersz w out
    if (ox < height && oy < width)
        out[oy * height + ox] = tile[threadIdx.x][threadIdx.y];
}

#define TILEx 32
#define TILEy 8
// in: height x width, out: width x height
__global__ void transpose_tiled_fewer_threads(const float* in, float* out, int width, int height) {
    __shared__ float tile[TILE][TILE+1];

    int x = blockIdx.x * TILE + threadIdx.x;        // kolumna w in
    int y = blockIdx.y * TILE + threadIdx.y;        // wiersz w in
    for (int i = 0; i < TILE / TILEy && x < width && y + i * TILEy < height; i++) {
        tile[threadIdx.y + i * TILEy][threadIdx.x] = in[(y + i * TILEy) * width + x];
    }
    __syncthreads();

    int ox = blockIdx.y * TILE + threadIdx.x;       // kolumna w out
    int oy = blockIdx.x * TILE + threadIdx.y;       // wiersz w out
    for (int i = 0; i < 4 && ox < height && oy + i * TILEy < width; i++) {
        out[(oy + i * TILEy) * height + ox] = tile[threadIdx.x][threadIdx.y + i * TILEy];
    }
}

// Uruchamia kernel 10 razy (po rozgrzewce), wypisuje średni czas i GB/s
void run(const char* name, KernelFn k, const float* d_in, float* d_out, int width, int height) {
  dim3 block(TILEx, TILEy);
  dim3 grid((width + TILE - 1) / TILE, (height + TILE - 1) / TILE);
  k<<<grid, block>>>(d_in, d_out, width, height);          // rozgrzewka
  CHECK(cudaGetLastError());
  CHECK(cudaDeviceSynchronize());

  cudaEvent_t start, stop; cudaEventCreate(&start); cudaEventCreate(&stop);
  cudaEventRecord(start);
  for (int r = 0; r < 10; ++r) k<<<grid, block>>>(d_in, d_out, width, height);
  cudaEventRecord(stop);
  CHECK(cudaEventSynchronize(stop));
  float ms; cudaEventElapsedTime(&ms, start, stop); ms /= 10;

  double gbs = 2.0 * width * height * sizeof(float) / ms / 1e6;   // odczyt + zapis
  printf("%-10s %8.3f ms  %7.1f GB/s\n", name, ms, gbs);
  cudaEventDestroy(start); cudaEventDestroy(stop);
}

// Sprawdza, czy out jest transpozycją in
bool check_transpose(const std::vector<float>& in, const std::vector<float>& out, int width, int height) {
  for (int y = 0; y < height; ++y)
    for (int x = 0; x < width; ++x)
      if (out[(size_t)x * height + y] != in[(size_t)y * width + x]) {
        printf("  pierwszy błąd przy in[%d][%d]\n", y, x);
        return false;
      }
  return true;
}

int main(int argc, char** argv) {
  int width  = argc > 2 ? atoi(argv[1]) : 2048;
  int height = argc > 2 ? atoi(argv[2]) : 2048;
  size_t count = (size_t)width * height, bytes = count * sizeof(float);

  std::vector<float> h_in(count), h_out(count);
  for (auto& v : h_in) v = rand() / (float)RAND_MAX;

  float *d_in, *d_out;
  CHECK(cudaMalloc(&d_in, bytes));
  CHECK(cudaMalloc(&d_out, bytes));
  CHECK(cudaMemcpy(d_in, h_in.data(), bytes, cudaMemcpyHostToDevice));
  printf("Macierz %d x %d (szerokość x wysokość)\n", width, height);

  // kopia: sprawdzamy, czy out == in
  CHECK(cudaMemset(d_out, 0, bytes));
  run("kopia", copy_kernel, d_in, d_out, width, height);
  CHECK(cudaMemcpy(h_out.data(), d_out, bytes, cudaMemcpyDeviceToHost));
  printf("  wynik: %s\n", h_out == h_in ? "OK" : "ZŁY");

  // transpozycje: sprawdzamy z wersją CPU
  KernelFn kernels[] = { transpose_naive, transpose_tiled };
  const char* names[] = { "naiwna", "kafelek" };
  for (int k = 0; k < 2; ++k) {
    CHECK(cudaMemset(d_out, 0, bytes));   // żeby stary wynik nie udawał poprawnego
    run(names[k], kernels[k], d_in, d_out, width, height);
    CHECK(cudaMemcpy(h_out.data(), d_out, bytes, cudaMemcpyDeviceToHost));
    printf("  wynik: %s\n", check_transpose(h_in, h_out, width, height) ? "OK" : "ZŁY");
  }
  cudaFree(d_in); cudaFree(d_out);
}
Writing transpose.cu